# **History aware bot**

In [1]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

In [11]:
db_path = "db/chroma_db"
embedding_model_path = "../../../Data/HuggingFaceEmbeddings Models/all-mpnet-base-v2"

MAX_CHARS = 2000

In [4]:
# Loading embedding model

embedding_model = HuggingFaceEmbeddings(
    model_name=embedding_model_path,
    model_kwargs={"device" : "cpu"},
    encode_kwargs={"normalize_embeddings":True}
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [5]:
# COnnect to DB

# since here Chroma(...) is the constructor for opening or creating a vector store, we passs embedding model as function to use when retreiving
db = Chroma(
    persist_directory=db_path,
    embedding_function=embedding_model
)

### **LLM**

In [6]:
llm = ChatOllama(model="llama3")

In [ ]:
# Store chat coversations as msgs
chat_history = []

In [ ]:
def ask_question(user_question, intent):

    recent_history = chat_history[-6:]

    # Question updating
    if recent_history:

        # prompt for question updating
        question_prompt= [
            SystemMessage(content="""
                    Given the conversation history and the latest user message,
                    rewrite the latest message into a complete standalone search query.
                    Preserve the original meaning.
                    Do not answer the question.
                    Return only the rewritten query.
                """)
        ] + recent_history + [
            HumanMessage(content=f"New question : {user_question}")
        ]

        # new question genration
        print("#### new question genrating...")
        updated_question = llm.invoke(question_prompt).content.strip()

    else :
        updated_question = user_question

    # Find relevant docs for updated questoin
    retreiver = db.as_retriever(search_kwargs={"k":3})
    relevant_docs = retreiver.invoke(updated_question)
    print("Relevant docs found")

    # combined all relevant docs as pointwise single string
    combined_docs = "\n".join([f" - {doc.page_content}" for doc in relevant_docs])
    
    final_msg = f"""
        User message:
        {user_question}

        Rewritten query (for context):
        {updated_question}

        Intent:
        {intent}
        
        Documents:
        {combined_docs}

        Instruction:
        Generate a helpful banking response based on the User message, Intent & Documents. If Intent is 'unknown' dont care about it. 
        If you can't find helpfull answer in docs say I'm wasn't able to find any information please contact support.
    """

    final_msg = final_msg[:MAX_CHARS]

    final_prompt = [
        SystemMessage(content="You are a helpful assistant that answer questions based on provided documents and conversation hostory. Answer should be less thatn 200 chars.")
    ]+ recent_history + [
        HumanMessage(content=final_msg)
    ]

    print("Generating answer...")
    response = llm.invoke(final_prompt).content.strip()

    print("-"*100)
    print(f"### Response: {response} ***")
    print("-"*100)

    chat_history.append(HumanMessage(content=user_question))
    chat_history.append(AIMessage(content=response))

    return response

In [13]:
def start_chat():
    print("Ask me question : type 'quit' to exit")
    intent = "unknown"
    
    while True:
        question= input("AsK : ")
        if question.lower() == 'quit':
            print("Thank you")
            break
        ask_question(question, intent)

In [ ]:
chat_history=[]
start_chat()

Ask me question : type 'quit' to exit


AsK :  what is the name of bank?


Relevant docs found
Generating answer...
----------------------------------------------------------------------------------------------------
### Response: The name of the bank is Commercial Bank of Ceylon PLC. ***
----------------------------------------------------------------------------------------------------


AsK :  last year revenue?


#### new question genrating...
Relevant docs found
Generating answer...
----------------------------------------------------------------------------------------------------
### Response: According to the Annual Report, the last year revenue was not specified. The report only mentions that it is for the financial year ended December 31, 2025, but does not provide the actual revenue figure. ***
----------------------------------------------------------------------------------------------------


AsK :  quit


Thank you
